# SmartHome — entrenamiento YOLO de clasificación
Clases: `auto_dueno` y `otro_auto`. El modelo clasifica el recorte generado por OpenCV.


In [4]:
!pip install -q -U ultralytics
from ultralytics import YOLO
from google.colab import files
from pathlib import Path
import zipfile, shutil, json, os


## 1. Subir `dataset_yolo_clasificacion.zip`


In [5]:
subidos = files.upload()
zip_name = next(iter(subidos))
destino = Path('/content/datos_smarthome')
if destino.exists(): shutil.rmtree(destino)
destino.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as zf: zf.extractall(destino)
candidatos = [p for p in destino.rglob('*') if p.is_dir() and (p/'train').exists() and (p/'val').exists()]
assert candidatos, 'No se encontró la carpeta con train/val/test'
DATASET = candidatos[0]
print('Dataset:', DATASET)
for split in ['train','val','test']:
    print(split, {c.name: len(list(c.glob('*'))) for c in (DATASET/split).iterdir() if c.is_dir()})


Saving dataset_yolo_clasificacion_ligero.zip to dataset_yolo_clasificacion_ligero.zip
Dataset: /content/datos_smarthome/dataset_yolo_clasificacion_ligero
train {'otro_auto': 1088, 'auto_dueno': 461}
val {'otro_auto': 311, 'auto_dueno': 132}
test {'otro_auto': 156, 'auto_dueno': 67}


## 2. Entrenar modelo pequeño de clasificación


In [6]:
model = YOLO('yolo11n-cls.pt')
resultado = model.train(
    data=str(DATASET),
    epochs=40,
    imgsz=224,
    batch=32,
    patience=8,
    device=0,
    workers=2,
    project='/content/runs_smarthome',
    name='auto_dueno_cls',
    seed=247,
)


Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datos_smarthome/dataset_yolo_clasificacion_ligero, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=auto_dueno_cls, nbs=64, nms=False, opset=None, optimize=False, optimizer

## 3. Validar y revisar predicciones


In [7]:
BEST = Path('/content/runs_smarthome/auto_dueno_cls/weights/best.pt')
assert BEST.exists(), BEST
mejor = YOLO(str(BEST))
metricas = mejor.val(data=str(DATASET), split='test', imgsz=224)
print('Modelo:', BEST)


Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
train: /content/datos_smarthome/dataset_yolo_clasificacion_ligero/train... found 1549 images in 2 classes ✅ 
val: /content/datos_smarthome/dataset_yolo_clasificacion_ligero/val... found 443 images in 2 classes ✅ 
test: /content/datos_smarthome/dataset_yolo_clasificacion_ligero/test... found 223 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1121.2±407.0 MB/s, size: 33.7 KB)
test: Scanning /content/datos_smarthome/dataset_yolo_clasificacion_ligero/test... 223 images, 0 corrupt: 100% ━━━━━━━━━━━━ 223/223 5.1Kit/s 0.0s
test: New cache created: /content/datos_smarthome/dataset_yolo_clasificacion_ligero/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 14/14 11.4it/s 1.2s
                   all          1          1
Speed: 0.8ms preprocess, 3.2ms inference, 0.0ms loss, 0.0m

In [8]:
muestras = []
for clase_dir in sorted((DATASET/'test').iterdir()):
    if clase_dir.is_dir(): muestras.extend(list(clase_dir.glob('*'))[:5])
preds = mejor.predict(source=[str(p) for p in muestras], imgsz=224, verbose=False)
for ruta, pred in zip(muestras, preds):
    i = int(pred.probs.top1)
    conf = float(pred.probs.top1conf.item())
    print(ruta.parent.name, '->', pred.names[i], f'{conf:.3f}', ruta.name)


auto_dueno -> auto_dueno 1.000 auto_dueno_00013.jpg
auto_dueno -> auto_dueno 0.998 auto_dueno_00062.jpg
auto_dueno -> auto_dueno 1.000 auto_dueno_00019.jpg
auto_dueno -> auto_dueno 1.000 auto_dueno_00055.jpg
auto_dueno -> auto_dueno 1.000 auto_dueno_00042.jpg
otro_auto -> otro_auto 1.000 otro_auto_00123.jpg
otro_auto -> otro_auto 0.999 otro_auto_00097.jpg
otro_auto -> otro_auto 1.000 otro_auto_00076.jpg
otro_auto -> otro_auto 0.987 otro_auto_00001.jpg
otro_auto -> otro_auto 1.000 otro_auto_00018.jpg


## 4. Descargar el modelo


In [9]:
salida = Path('/content/best_cls.pt')
shutil.copy2(BEST, salida)
files.download(str(salida))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>